In [2]:
import json
import re
import pickle
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [3]:
DATA_PATH = "../data/raw/movies_raw_enriched.csv"

movies_df = pd.read_csv(DATA_PATH)

print("Raw dataset shape:", movies_df.shape)

print("\nMissing values:")
print(movies_df.isnull().sum())

print(
    "\nDuplicate movie IDs:",
    movies_df["movie_id"].duplicated().sum()
)

print(
    "Duplicate titles:",
    movies_df["title"].duplicated().sum()
)

Raw dataset shape: (35001, 12)

Missing values:
movie_id           0
title              0
overview        3342
release_date      42
genres             0
keywords           0
runtime            0
vote_average       0
vote_count         0
poster_url      3549
cast            1658
director        1007
dtype: int64

Duplicate movie IDs: 0
Duplicate titles: 1665


In [9]:
movies = movies_df[
    [
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "director"
    ]
].copy()

print("\nSelected dataset shape:", movies.shape)

movies = movies.dropna().reset_index(drop=True)

print("After dropping missing values:", movies.shape)
print("Total movies:", len(movies))


Selected dataset shape: (35001, 7)
After dropping missing values: (29820, 7)
Total movies: 29820


In [10]:
# Feature weights: how many times each field is repeated in the text "soup"
# Recommendation based on the importance of genres, keywords, cast and director (you can vary according to your choice)
W_GENRES = 4
W_KEYWORDS = 1
W_CAST = 3
W_DIRECTOR = 3

In [11]:
def convert(obj):
    try:
        result = json.loads(obj)

        if isinstance(result, list):
            return result

        return []

    except (json.JSONDecodeError, TypeError):
        return []


movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)

In [12]:
def squash(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())

In [13]:
def build_text(row):

    genres = [
        squash(g)
        for g in row["genres"]
        if isinstance(g, str) and g.strip()
    ]

    keywords = [
        squash(k)
        for k in row["keywords"]
        if isinstance(k, str) and k.strip()
    ]

    cast = [
        squash(c)
        for c in row["cast"].split("|")[:5]
        if c.strip()
    ]

    director = [
        squash(d)
        for d in row["director"].split("|")
        if d.strip()
    ]

    return " ".join(
        [row["overview"].lower()]
        + genres * W_GENRES
        + keywords * W_KEYWORDS
        + cast * W_CAST
        + director * W_DIRECTOR
    )


movies["soup"] = movies.apply(build_text, axis=1)

In [14]:
new_df = movies[
    [
        "movie_id",
        "title",
        "soup"
    ]
].copy()

new_df = new_df.reset_index(drop=True)

print("\nFinal dataset shape:", new_df.shape)
print("Total movies:", len(new_df))


Final dataset shape: (29820, 3)
Total movies: 29820


In [15]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 1),
    min_df=2,
    max_features=100_000,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(new_df["soup"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (29820, 56336)


In [16]:
def recommend(title, movie_id=None, n=5):

    if not isinstance(title, str) or not title.strip():
        return "Please provide a valid movie title."

    if n <= 0:
        return new_df.iloc[0:0][
            ["movie_id", "title"]
        ].copy()

    key = title.strip().casefold()

    normalized_titles = (
        new_df["title"]
        .str.strip()
        .str.casefold()
    )

    matches = new_df.index[
        normalized_titles == key
    ].tolist()

    if not matches:

        close = [
            t
            for t in new_df["title"].dropna().unique()
            if key in t.strip().casefold()
        ][:5]

        return (
            f"'{title}' not found. "
            f"Similar titles: {close}"
        )

    if movie_id is not None:

        matches = [
            i
            for i in matches
            if str(new_df.iloc[i]["movie_id"])
            == str(movie_id)
        ]

        if not matches:
            return (
                f"Movie ID {movie_id} "
                f"does not match '{title}'."
            )

    movie_index = matches[0]

    selected_title = normalized_titles.iloc[movie_index]

    similarities = linear_kernel(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).ravel()

    candidates = new_df[
        ["movie_id", "title"]
    ].copy()

    candidates["similarity"] = similarities

    candidates = candidates[
        (candidates.index != movie_index)
        & (normalized_titles != selected_title)
    ].copy()

    results = candidates.sort_values(
        "similarity",
        ascending=False,
        kind="stable"
    ).head(n)

    return results[
        ["movie_id", "title"]
    ].reset_index(drop=True)

In [17]:
test_movies = [

    "The Dark Knight",
    "Iron Man",
    "Spider-Man",

    "Interstellar",
    "Inception",
    "The Matrix",

    "The Notebook",
    "Titanic",
    "La La Land",

    "3 Idiots",
    "Home Alone",
    "The Hangover",

    "Toy Story",
    "Finding Nemo",
    "Coco",

    "The Godfather",
    "Se7en",
    "Gone Girl",

    "The Conjuring",
    "A Nightmare on Elm Street",

    "Drishyam",
    "Dangal",
    "3 Idiots",

    "The Lord of the Rings",
    "Harry Potter and the Philosopher's Stone",

    "Shutter Island",
    "The Prestige"
]

In [18]:
for movie in test_movies:

    print(f"\n{'=' * 60}")
    print(f"Recommendations for: {movie}")
    print("=" * 60)

    result = recommend(movie)

    if isinstance(result, str):
        print(result)
    else:
        print(result.to_string(index=False))


Recommendations for: The Dark Knight
 movie_id                 title
      272         Batman Begins
    49026 The Dark Knight Rises
     1124          The Prestige
      268                Batman
   414906            The Batman

Recommendations for: Iron Man
 movie_id                     title
    10138                Iron Man 2
    68721                Iron Man 3
   315635    Spider-Man: Homecoming
    99861   Avengers: Age of Ultron
   429617 Spider-Man: Far From Home

Recommendations for: Spider-Man
 movie_id                                       title
      559                                Spider-Man 3
   969681                   Spider-Man: Brand New Day
     1930                      The Amazing Spider-Man
   315635                      Spider-Man: Homecoming
   453395 Doctor Strange in the Multiverse of Madness

Recommendations for: Interstellar
 movie_id                 title
   286217           The Martian
  1368337           The Odyssey
     1124          The Prestige
  1

In [19]:
MODEL_DIR = Path("../backend")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_artifacts = {
    "movies": new_df.reset_index(drop=True),
    "tfidf_vectorizer": tfidf,
    "tfidf_matrix": tfidf_matrix,
    "version": "v2"
}

model_path = MODEL_DIR / "movie_recommender.pkl"

with open(model_path, "wb") as file:
    pickle.dump(
        model_artifacts,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print(f"\nV2 model saved to: {model_path.resolve()}")
print(f"Movies: {len(new_df):,}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(
    f"File size: "
    f"{model_path.stat().st_size / (1024**2):.2f} MB"
)


V2 model saved to: D:\Seenema Final\Seenema\backend\movie_recommender.pkl
Movies: 29,820
TF-IDF matrix shape: (29820, 56336)
File size: 30.21 MB
